# Day 1 · Your first LLM call & reading its data

*Day 1 — Linguistic Data Analysis II*

### How to use this notebook

This is your **single submission for the day**. It has two parts:

- **Part A · Tutorial** — call a language model, then learn to read the data it hands back.
- **Part B · Corpus Lab** — segment text into sentences, run the model over a list, then practice the basics.

You only edit the cells marked **✏️ YOU EDIT**. Cells marked **🔧 Library cell** are pre-written — run them, don't change them.

➡️ Work top to bottom. When you're done, **Runtime → Run all**, then **File → Download → Download `.ipynb`** and submit that file.

## Part A · Tutorial — your first LLM call, and reading its answer

No prior Python needed. The star of Part A is a single call to a language model; everything else — variables, data types, f-strings — is just enough Python to read and reshape what the model gives you back. Run each cell, read the output, then change a value and re-run to see what happens.

### 1. Getting your bearings in Colab

This page is a **Colab notebook**: a stack of **cells** you run top to bottom. A code cell runs when you press **Shift+Enter** (or click ▶). The first run wakes up a **runtime** — a temporary computer in the cloud that remembers your variables until you close the tab. Run the cell below to prove it works.

In [ ]:
print("Hello, Colab! You just ran your first cell.")

**When a cell breaks — read the error.** Sooner or later a cell turns red. Don't panic: Python tells you what went wrong on the **last line**. For example, running `print(mesage)` (a typo for `message`) prints:

```
NameError: name 'mesage' is not defined
```

Read it back to front: the **error type** (`NameError`), the **message** (`'mesage' is not defined`), and the **line** it happened on. Nearly every early error is a typo or a cell you haven't run yet. Fix the typo, re-run, move on.

### 2. Your first LLM call

Run the setup cell (it loads the LLM backend — in Colab that's the free built-in Gemini, no key needed), then send the model a prompt with `generate_text(...)`. The answer comes back as text, which we store in a **variable** called `reply`.

In [ ]:
#@title 📦 Setup — run me first { display-mode: "form" }
# Imports + the LLM backend. No pip install needed in Colab.
import json, re, time, urllib.request, os
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

MODEL_ID = "gemini-3.1-flash-lite"   # pinned model for the reproducible (API) backend

def _resolve_gemini_key():
    """Find a Gemini API key: Colab Secrets first (not auto-exported to env), then env."""
    try:
        from google.colab import userdata      # only exists in Colab
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass                                    # not in Colab, or secret not set
    return os.environ.get("GEMINI_API_KEY")

def _make_api_backend(key):
    """Reproducible backend: Gemini API with temperature=0 + a fixed seed."""
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=key)
    cfg = types.GenerateContentConfig(temperature=0, seed=42)
    return (lambda p: client.models.generate_content(
                model=MODEL_ID, contents=p, config=cfg).text,
            f"Gemini API ({MODEL_ID}, temperature=0, seed=42)")

# ---- keeps us from calling the model faster than the free tier allows ----------
# (explained step by step in Day 3, right after this cell — the short version:
#  wait a bit between calls, and if we still get told to slow down, wait longer
#  and try again, unless the message says we are out of quota for the whole day.)
def _looks_like_rate_limit(error):
    text = str(error).lower()
    return any(s in text for s in
               ["429", "resource_exhausted", "rate limit", "quota", "too many requests"])

def _looks_like_daily_quota(error):
    text = str(error).lower()
    return "per day" in text or "perday" in text.replace(" ", "")

def _suggested_delay(error, fallback):
    m = re.search(r"retry in ([0-9]+(?:\.[0-9]+)?)s", str(error).lower())
    return float(m.group(1)) + 1.0 if m else fallback

_last_call_time = 0.0   # generate_text remembers & updates this with `global`

def generate_text(prompt, max_retries=5):
    global _last_call_time
    for attempt in range(max_retries + 1):
        wait = _min_interval - (time.monotonic() - _last_call_time)
        if wait > 0:
            time.sleep(wait)
        try:
            _last_call_time = time.monotonic()
            return _raw_generate_text(prompt)
        except Exception as error:
            if not _looks_like_rate_limit(error):
                raise                                   # a real bug — don't hide it
            if _looks_like_daily_quota(error):
                raise RuntimeError(
                    "Daily quota used up for today — waiting won't help until it "
                    "resets. Come back tomorrow, or ask your instructor.") from error
            if attempt == max_retries:
                raise
            print(f"  (rate limited — waiting before trying again, attempt {attempt+1})")
            time.sleep(_suggested_delay(error, _min_interval * (attempt + 2)))
    raise RuntimeError("Still rate-limited after several tries.")

# Prefer the API key when set (reproducible); else fall back to colab.ai (demo).
_key = _resolve_gemini_key()
if _key:
    _raw_generate_text, _backend = _make_api_backend(_key)
    _min_interval = 4.4     # keeps us under gemini-3.1-flash-lite's 15-requests/minute cap
else:
    try:
        from google.colab import ai            # Colab's built-in Gemini — no key
        _raw_generate_text, _backend = (lambda p: ai.generate_text(p)), "Colab Gemini (demo, non-reproducible)"
        _min_interval = 13.2   # colab.ai publishes no rate limit — pace conservatively
    except ImportError:
        raise RuntimeError(
            "No LLM backend found. Run this notebook in Google Colab (free built-in "
            "Gemini, no key needed), or set GEMINI_API_KEY — in Colab via the Secrets "
            "panel, or as an environment variable when running locally. "
            "See resources/tools/gemini-api-key.md.")
print(f"Setup done. LLM backend: {_backend}. scikit-learn ready.")

**✏️ YOU EDIT** — change the prompt text and re-run. The prompt is just text you send; the reply is just text you get back.

In [ ]:
reply = generate_text("In one sentence, what is applied linguistics?")
print(reply)

### 3. What did the model hand back? — Python data types

Every value in Python has a **type**. Ask what type `reply` is:

In [ ]:
print(type(reply))     # <class 'str'> — a string, i.e. text

The three types you'll use all week:

- **`str`** — text, in quotes (like `reply`).
- **`list`** — an ordered sequence, in square brackets. Several sentences, say.
- **`dict`** — a labelled record, in curly braces: `key → value` pairs. This is the `{id, text, label}` shape every dataset this week uses.

Build one of each, then reach inside them with **`[...]`** — lists by position (counting from 0), dicts by key.

In [ ]:
sentences = ["The cat sat on the mat.",          # a list of strings
             "Nevertheless, the findings were inconclusive."]
record = {"id": 1, "text": sentences[0], "label": "A1"}   # a dict

print("how many sentences:", len(sentences))
print("first sentence:", sentences[0])       # list — index by position (from 0)
print("its gold label:", record["label"])   # dict — index by key

### 4. f-strings — put your data *into* a prompt

An **f-string** (`f"..."`) drops a variable straight into a piece of text with `{curly braces}`. That's how you build a prompt *about* a specific sentence — the trick we'll use to run one prompt over a whole dataset later.

**✏️ YOU EDIT** — change the sentence and re-run.

In [ ]:
sentence = "Nevertheless, the findings were inconclusive."
prompt = f"What CEFR level (A1-C2) is this sentence? {sentence}"
print("Prompt sent:", prompt)

reply = generate_text(prompt)
print("Model says:", reply)

## Part B · Corpus Lab — from text to sentences, then practice

In Part A you called the model once. Here you'll turn a paragraph into individual **sentences** (the unit you'll annotate on Day 2), run the model over all of them, and then practice the basics on your own. Cells marked **✏️ YOU EDIT** are yours to change; run the **self-check** at the end until every line prints ✅.

### 1. Splitting text into sentences — *without* a model

Text arrives as one long string. To analyse it sentence by sentence you first have to **segment** it. The obvious idea: split on the full stop. To do that we call a **method** on the string — `some_text.split(".")` — using a dot (`.`) to run a built-in action on a value.

In [ ]:
paragraph = ("Dr. Smith reviewed the data. The results were clear, e.g. accuracy rose. Scores went from 3.14 to 9.")

naive = paragraph.split(".")     # cut the string at every "."
print("pieces:", len(naive))
for piece in naive:
    print(repr(piece))

Look at the output: `"Dr"` (from *Dr.*), `"e"` and `"g"` (from *e.g.*), and `"3"`/`"14"` (from *3.14*) all got split in the wrong places. **Sentence boundaries are not just full stops** — abbreviations and decimals break the naive rule.

### 2. Splitting text into sentences — *with* a model

A proper tool knows more than "cut at every dot". We'll use **spaCy**, an NLP library. `import spacy` loads that toolbox; `spacy.blank("en")` makes a minimal English pipeline and we add a rule-based **sentencizer** to it (no model download needed).

In [ ]:
import spacy
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

doc = nlp(paragraph)
sentences = list(doc.sents)        # spaCy's sentence objects, as a list
print("sentences found:", len(sentences))
print("first sentence:", sentences[0])

spaCy keeps `Dr.` and `e.g.` intact and finds the real boundaries. **Why this matters:** on Day 2 the unit you annotate and feed the LLM is the *sentence* — bad boundaries mean bad data downstream.

### 3. Run the model over every sentence — `for`, `if`, and a function

Now that you have a list of sentences, do something to *each* one. A **`for` loop** repeats the same steps for every item; an **`if`** lets you react to what comes back. Below, we build a prompt for each sentence (with an f-string) and ask the model for its CEFR level.

**✏️ YOU EDIT** — try your own sentences.

In [ ]:
examples = ["The findings were inconclusive.",
            "Nevertheless, we draw some tentative conclusions.",
            "More research is needed."]

for sentence in examples:
    prompt = f"What CEFR level (A1-C2) is this sentence? Answer with just the level. {sentence}"
    reply = generate_text(prompt).strip()
    if reply == "":
        print(sentence, "→ (no answer)")
    else:
        print(sentence, "→", reply)

Finally, wrap those three steps — build prompt, call model, tidy the reply — into a **function** you can reuse. Defining `ask(...)` once means the rest of the week you just call `ask(sentence)`. This little function is the seed of the pipeline you'll assemble later.

In [ ]:
def ask(sentence):
    """Ask the model for the CEFR level of one sentence; return its reply."""
    prompt = f"What CEFR level (A1-C2) is this sentence? Answer with just the level. {sentence}"
    return generate_text(prompt).strip()

print(ask("The cat sat on the mat."))

### 4. Your turn — Python practice

Fill in each function so it does what its docstring says (replace the `raise NotImplementedError(...)` line). Then run the **self-check** cell below until every line prints ✅. No grader needed — the checks *are* your grader.

In [ ]:
# ✏️ YOU EDIT — replace each NotImplementedError with your code.

def label_of(item):
    """Return the value stored under the key "label" in the dict `item`.
    Example: label_of({"id": 1, "text": "Hi", "label": "A1"}) -> "A1".
    """
    raise NotImplementedError("Return item['label'].")


def long_words(words, n):
    """Return a LIST of the words whose length is greater than n.
    Example: long_words(["a", "cat", "elephant"], 3) -> ["elephant"].
    """
    # HINT: build a result list; loop with `for w in words:`; keep w if len(w) > n.
    raise NotImplementedError("Return the words longer than n characters.")


def count_labels(items):
    """Given a list of {id, text, label} dicts, return a dict mapping each
    label to how many times it appears.
    Example: count_labels([{"label":"A1"}, {"label":"A1"}, {"label":"B1"}])
             -> {"A1": 2, "B1": 1}.
    """
    # HINT: start with counts = {}; for each item, add 1 to counts[label]
    #       (use counts.get(label, 0) + 1 so the first time starts at 0).
    raise NotImplementedError("Count how many items carry each label.")

In [ ]:
#@title 🔎 Self-check — run me { display-mode: "form" }
sample = [{"id": 1, "text": "Hi.", "label": "A1"},
          {"id": 2, "text": "Hello there.", "label": "A1"},
          {"id": 3, "text": "Nevertheless...", "label": "C1"}]
checks = [
    ("label_of", label_of(sample[0]) == "A1"),
    ("long_words", long_words(["a", "cat", "elephant"], 3) == ["elephant"]),
    ("count_labels", count_labels(sample) == {"A1": 2, "C1": 1}),
]
for name, ok in checks:
    print(("✅" if ok else "❌"), name)
print("All passed ✅" if all(ok for _, ok in checks)
      else "Some checks failed — fix them and re-run.")

### Python you'll *see* but won't have to write

The pre-written **🔧 Library cells** later this week occasionally use two shorthands. You never have to write them — just recognise them:

- A **list comprehension** builds a list in one line. `[row["label"] for row in rows]` means "the `label` of every row" — the same as a `for` loop that `.append`s each label.
- **`try` / `except`** (handle an error instead of crashing) and **`global`** (let a function update a shared variable) show up in the LLM backend and are explained on **Day 3** — no need to learn them today.

---
## Mini-project — form your group & pick a track

Before Day 2 you'll settle into a project group and choose a dataset **track**. Each group needs **at least one *Linguistic Data Analysis I* alumnus**. Once formed, pick your track and note it — you'll annotate that dataset tomorrow.

See the [Final Project](../final-project/index.md) page for the tracks and what the project involves.

---
## ✅ Before you submit

1. **Runtime → Run all** and check every cell ran without error.
2. Tutorial outputs are visible (tables / charts / the model's answers).
3. Every Corpus Lab self-check prints ✅ (or your TODO answers are filled in).
4. **File → Download → Download `.ipynb`** and upload that one file.